In [1]:

# # ── Inspect test-split videos: duration / fps / resolution ────────────
# import subprocess, json, re
# from pathlib import Path

# test_dir = Path.cwd() / "MELD.Raw" / "test" / "output_repeated_splits_test"
# videos = sorted(test_dir.glob("*.mp4"))
# print(f"Test videos found: {len(videos)}")

# STANDARD_RE = re.compile(r'^dia\d+_utt\d+\.mp4$')

# rows = []
# for p in videos:
#     is_standard = bool(STANDARD_RE.match(p.name))
#     try:
#         r = subprocess.run(
#             ["ffprobe", "-v", "quiet", "-print_format", "json",
#              "-show_streams", "-select_streams", "v:0", str(p)],
#             capture_output=True, text=True, timeout=5,
#         )
#         streams = json.loads(r.stdout).get("streams", [])
#         if streams:
#             s = streams[0]
#             dur  = float(s.get("duration", 0) or 0)
#             w, h = s.get("width"), s.get("height")
#             rn, rd = s.get("r_frame_rate", "0/1").split("/")
#             fps  = round(int(rn) / max(int(rd), 1), 3)
#             nf   = s.get("nb_frames", "?")
#             rows.append({"name": p.name, "standard": is_standard, "dur": dur, "fps": fps,
#                          "w": w, "h": h, "nb_frames": nf, "size_mb": round(p.stat().st_size / 1e6, 3)})
#         else:
#             rows.append({"name": p.name, "standard": is_standard, "dur": 0.0, "fps": None,
#                          "w": None, "h": None, "nb_frames": "no-stream", "size_mb": round(p.stat().st_size / 1e6, 3)})
#     except Exception as e:
#         rows.append({"name": p.name, "standard": is_standard, "dur": -1, "fps": None,
#                      "w": None, "h": None, "nb_frames": str(e), "size_mb": 0})

# import pandas as pd
# df_inspect = pd.DataFrame(rows).sort_values("dur", ascending=False)

# # ── Naming summary ─────────────────────────────────────────────────────
# std_count   = df_inspect["standard"].sum()
# nonst_count = (~df_inspect["standard"]).sum()
# print(f"\nNaming:  standard={std_count}  non-standard={nonst_count}")

# # Show non-standard names (first 10)
# nonstandard = df_inspect[~df_inspect["standard"]]
# print(f"\nNon-standard filenames ({len(nonstandard)} total) — first 10:")
# print(nonstandard["name"].head(10).to_string(index=False))

# # ── Duration summary ───────────────────────────────────────────────────
# durs = df_inspect[df_inspect["dur"] > 0]["dur"]
# print(f"\nDuration stats ({len(durs)} videos with valid duration):")
# print(f"  Max:    {durs.max():.2f}s")
# print(f"  Min:    {durs.min():.2f}s")
# print(f"  Mean:   {durs.mean():.2f}s")
# print(f"  Median: {durs.median():.2f}s")

# bins   = [0, 5, 10, 20, 30, 60, float("inf")]
# labels = ["≤5s", "5-10s", "10-20s", "20-30s", "30-60s", ">60s"]
# print("\nDistribution:")
# for lo, hi, lbl in zip(bins, bins[1:], labels):
#     n = ((durs > lo) & (durs <= hi)).sum()
#     print(f"  {lbl:>8}: {n}")

# no_dur = df_inspect[df_inspect["dur"] <= 0]
# print(f"\nVideos with no/zero duration: {len(no_dur)}")

# print("\nTop 20 longest:")
# print(df_inspect[["name", "standard", "dur", "fps", "w", "h", "nb_frames", "size_mb"]].head(20).to_string(index=False))


In [2]:

import os
import torch
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info

BASE_DIR = Path.cwd()
MELD_DIR = BASE_DIR / "MELD.Raw"
MANIFEST_PATH = MELD_DIR / "meld_vlm_manifest.csv"

print("Imports OK")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


2026-03-02 16:57:44.951927: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-02 16:57:45.166217: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Imports OK
CUDA available: True
GPU: NVIDIA GeForce RTX 3060
VRAM: 12.5 GB


In [3]:

MODEL_ID     = "KlingTeam/VidEmo-3B"
PROCESSOR_ID = "Qwen/Qwen2.5-VL-3B-Instruct"

print(f"Loading model: {MODEL_ID}")
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    # attn_implementation="flash_attention_2",  # enable if flash-attn is installed
)
model.eval()

processor = AutoProcessor.from_pretrained(PROCESSOR_ID)
# Decoder-only models require left-padding for correct batched generation
processor.tokenizer.padding_side = "left"

print("Model loaded successfully")


`torch_dtype` is deprecated! Use `dtype` instead!


Loading model: KlingTeam/VidEmo-3B


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. Note that this behavior will be extended to all models in a future release.


Model loaded successfully


In [4]:

# ── Build manifest from train / dev / test CSVs ───────────────────────
# Video path conventions (test dir has two naming styles):
#   standard : dia{D}_utt{U}.mp4
#   prefixed : final_videos_testdia{D}_utt{U}.mp4

SPLIT_CFG = {
    "train": {
        "csv":     MELD_DIR / "train" / "train_sent_emo.csv",
        "vid_dir": MELD_DIR / "train" / "train_splits",
    },
    "dev": {
        "csv":     MELD_DIR / "dev_sent_emo.csv",
        "vid_dir": MELD_DIR / "dev" / "dev_splits_complete",
    },
    "test": {
        "csv":     MELD_DIR / "test_sent_emo.csv",
        "vid_dir": MELD_DIR / "test" / "output_repeated_splits_test",
    },
}


def resolve_video_path(vid_dir: Path, dia: int, utt: int) -> str:
    """Return path to clip, trying standard name then the final_videos_test prefix."""
    for stem in (
        f"dia{dia}_utt{utt}.mp4",
        f"final_videos_testdia{dia}_utt{utt}.mp4",
    ):
        p = vid_dir / stem
        if p.exists():
            return str(p)
    return ""   # genuinely missing


frames = []
for split, cfg in SPLIT_CFG.items():
    df = pd.read_csv(cfg["csv"])
    df = df.rename(columns={"Sr No.": "sr_no"})
    df["split"] = split
    df["path"]  = df.apply(
        lambda r: resolve_video_path(cfg["vid_dir"], int(r["Dialogue_ID"]), int(r["Utterance_ID"])),
        axis=1,
    )
    frames.append(df)

manifest = pd.concat(frames, ignore_index=True)
manifest["vlm_analysis"] = pd.NA

# Drop rows whose video file wasn't resolved
missing = manifest["path"] == ""
print(f"Missing video files: {missing.sum()} (will be skipped)")
manifest = manifest[~missing].reset_index(drop=True)

print(f"\nManifest built:")
print(f"  Train : {(manifest['split']=='train').sum()}")
print(f"  Dev   : {(manifest['split']=='dev').sum()}")
print(f"  Test  : {(manifest['split']=='test').sum()}")
print(f"  Total : {len(manifest)}")
manifest.head(3)


Missing video files: 1 (will be skipped)

Manifest built:
  Train : 9989
  Dev   : 1108
  Test  : 2610
  Total : 13707


,sr_no,Utterance,Speaker,Emotion,Sentiment,Dialogue_ID,Utterance_ID,Season,Episode,StartTime,EndTime,split,path,vlm_analysis
0,1,also I was the point person on my companys tr...,Chandler,neutral,neutral,0,0,8,21,"00:16:16,059","00:16:21,731",train,/mnt/Work/ML/Code/EmoRecVid/MELD.Raw/train/tra...,<NA>
1,2,You mustve had your hands full.,The Interviewer,neutral,neutral,0,1,8,21,"00:16:21,940","00:16:23,442",train,/mnt/Work/ML/Code/EmoRecVid/MELD.Raw/train/tra...,<NA>
2,3,That I did. That I did.,Chandler,neutral,neutral,0,2,8,21,"00:16:23,442","00:16:26,389",train,/mnt/Work/ML/Code/EmoRecVid/MELD.Raw/train/tra...,<NA>


In [5]:

# ── Save initial manifest (or reload if resuming) ─────────────────────
if MANIFEST_PATH.exists():
    saved = pd.read_csv(MANIFEST_PATH)
    # Merge already-computed analyses back in
    manifest = manifest.merge(
        saved[["path", "vlm_analysis"]].rename(columns={"vlm_analysis": "_saved"}),
        on="path", how="left",
    )
    manifest["vlm_analysis"] = manifest["_saved"].combine_first(manifest["vlm_analysis"])
    manifest = manifest.drop(columns=["_saved"])
    done_before = manifest["vlm_analysis"].notna().sum()
    print(f"Resumed from existing manifest — {done_before}/{len(manifest)} already done.")
else:
    manifest.to_csv(MANIFEST_PATH, index=False)
    print(f"New manifest saved → {MANIFEST_PATH}")

manifest.head(3)


Resumed from existing manifest — 13707/13707 already done.


,sr_no,Utterance,Speaker,Emotion,Sentiment,Dialogue_ID,Utterance_ID,Season,Episode,StartTime,EndTime,split,path,vlm_analysis
0,1,also I was the point person on my companys tr...,Chandler,neutral,neutral,0,0,8,21,"00:16:16,059","00:16:21,731",train,/mnt/Work/ML/Code/EmoRecVid/MELD.Raw/train/tra...,The video begins with both individuals maintai...
1,2,You mustve had your hands full.,The Interviewer,neutral,neutral,0,1,8,21,"00:16:21,940","00:16:23,442",train,/mnt/Work/ML/Code/EmoRecVid/MELD.Raw/train/tra...,"The man begins with a neutral expression, his ..."
2,3,That I did. That I did.,Chandler,neutral,neutral,0,2,8,21,"00:16:23,442","00:16:26,389",train,/mnt/Work/ML/Code/EmoRecVid/MELD.Raw/train/tra...,The individual begins with a neutral expressio...


In [6]:

total       = len(manifest)
done_before = manifest["vlm_analysis"].notna().sum()
print(f"Total clips : {total}")
print(f"Already done: {done_before}  |  Remaining: {total - done_before}")

# ── Config ────────────────────────────────────────────────────────────
BATCH_SIZE    = 2      # clips per batch (reduce if OOM on normal clips)
SAVE_EVERY    = 4      # checkpoint frequency
FPS           = 1.0    # target fps for normal clips
MAX_FRAMES    = 64     # hard frame cap per clip
# Clips exceeding EITHER threshold are permanently skipped.
SKIP_DUR_S    = 60     # skip if duration  > this many seconds
SKIP_SIZE_MB  = 15     # skip if file size > this many MB

PROMPT_TEXT = (
    "Watch this short video clip of a person speaking and describe how their emotional state evolves over time.\n"
    "Structure your response as a temporal progression — divide the clip into beginning, middle, and end "
    "(or more segments if the emotion changes more than once).\n"
    "For each segment describe:\n"
    "- **Facial Expressions**: Specific muscle movements (brow, lips, eyes, jaw, cheeks).\n"
    "- **Head & Gaze**: Tilts, nods, shakes, eye direction.\n"
    "- **Body Language**: Posture shifts, gestures, tension or relaxation.\n"
    "- **Emotion at this moment**: The most likely emotion and the visual cues supporting it.\n"
    "Finish with a one-sentence summary of the overall emotional arc "
    "(e.g., starts neutral → builds frustration → brief smile at end).\n"
    "Ground every observation in a specific visible signal."
)


def get_video_info(path: str) -> tuple[float, float]:
    """Return (duration_s, size_mb). Uses stream then format fallback."""
    import subprocess, json, os
    size_mb = os.path.getsize(path) / 1e6
    try:
        r = subprocess.run(
            ["ffprobe", "-v", "quiet", "-print_format", "json",
             "-show_streams", "-select_streams", "v:0", path],
            capture_output=True, text=True, timeout=10,
        )
        streams = json.loads(r.stdout).get("streams", [])
        if streams:
            dur = float(streams[0].get("duration", 0) or 0)
            if dur > 0:
                return dur, size_mb
        # fallback: container-level duration
        r2 = subprocess.run(
            ["ffprobe", "-v", "quiet", "-print_format", "json", "-show_format", path],
            capture_output=True, text=True, timeout=10,
        )
        fmt_dur = float(json.loads(r2.stdout).get("format", {}).get("duration", 0) or 0)
        return fmt_dur, size_mb
    except Exception:
        pass
    return 0.0, size_mb


def is_playable(path: str) -> tuple[bool, str]:
    """
    Check that the file has a valid video stream with non-zero fps and
    that the container is intact (moov atom present).
    Returns (ok, reason_string).
    """
    import subprocess, json
    # 1. Check container integrity & fps via full probe
    r = subprocess.run(
        ["ffprobe", "-v", "error", "-print_format", "json",
         "-show_streams", "-show_format", path],
        capture_output=True, text=True, timeout=10,
    )
    stderr = r.stderr.lower()
    if "moov atom not found" in stderr or "invalid data found" in stderr:
        return False, f"corrupt: {r.stderr.strip()[:120]}"

    try:
        info    = json.loads(r.stdout)
        streams = info.get("streams", [])
        vid     = next((s for s in streams if s.get("codec_type") == "video"), None)
        if vid is None:
            return False, "no video stream"
        rn, rd = vid.get("r_frame_rate", "0/1").split("/")
        fps    = int(rn) / max(int(rd), 1)
        if fps <= 0:
            return False, f"fps={fps}"
    except Exception as e:
        return False, f"probe parse error: {e}"

    return True, ""


def safe_fps(path: str) -> float:
    """FPS that keeps total decoded frames ≤ MAX_FRAMES."""
    dur, _ = get_video_info(path)
    if dur <= 0:
        return FPS
    return round(min(FPS, MAX_FRAMES / dur), 4)


def analyze_batch(video_paths: list[str], max_new_tokens: int = 512) -> list[str]:
    """Run VidEmo on a batch of video clips; returns one response string per clip."""
    all_messages = [
        [{"role": "user", "content": [
            {"type": "video", "video": vp, "fps": safe_fps(vp), "max_pixels": 360 * 420},
            {"type": "text",  "text": PROMPT_TEXT},
        ]}]
        for vp in video_paths
    ]

    texts = [
        processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        for msgs in all_messages
    ]

    image_inputs, video_inputs, video_kwargs = process_vision_info(
        all_messages, return_video_kwargs=True
    )
    if "fps" in video_kwargs and isinstance(video_kwargs["fps"], list):
        del video_kwargs["fps"]

    inputs = processor(
        text=texts,
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
        **video_kwargs,
    ).to(model.device)

    with torch.inference_mode():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
        )

    generated_ids_trimmed = [
        out_ids[len(in_ids):]
        for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
    ]
    responses = processor.batch_decode(
        generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
    )
    return [r.strip() for r in responses]


# ── Fix legacy "ERROR: video too long" labels → SKIP ──────────────────
legacy_too_long = manifest["vlm_analysis"].str.startswith("ERROR: video too long", na=False)
if legacy_too_long.sum():
    manifest.loc[legacy_too_long, "vlm_analysis"] = manifest.loc[legacy_too_long, "vlm_analysis"].str.replace(
        "^ERROR:", "SKIP:", regex=True
    )
    manifest.to_csv(MANIFEST_PATH, index=False)
    print(f"Re-labelled {legacy_too_long.sum()} legacy 'ERROR: video too long' → 'SKIP:'")

# ── Pre-screen: permanently skip oversized / corrupt / unplayable clips ─
pending = list(manifest[manifest["vlm_analysis"].isna()].index)
print(f"\nQueued for processing: {len(pending)} clips")

skipped   = 0
safe_pending = []
for idx in pending:
    vp = manifest.at[idx, "path"]
    dur, size_mb = get_video_info(vp)

    # Size / duration guard
    if dur > SKIP_DUR_S or size_mb > SKIP_SIZE_MB:
        reason = f"dur={dur:.1f}s" if dur > SKIP_DUR_S else f"size={size_mb:.1f}MB"
        manifest.at[idx, "vlm_analysis"] = f"SKIP: clip too large ({reason})"
        print(f"  SKIP (too large): {Path(vp).name}  [{reason}]")
        skipped += 1
        continue

    # Playability guard (catches moov-missing, zero-fps, corrupt files)
    ok, reason = is_playable(vp)
    if not ok:
        manifest.at[idx, "vlm_analysis"] = f"SKIP: unplayable ({reason})"
        print(f"  SKIP (unplayable): {Path(vp).name}  [{reason}]")
        skipped += 1
        continue

    safe_pending.append(idx)

if skipped:
    manifest.to_csv(MANIFEST_PATH, index=False)
    print(f"Permanently skipped {skipped} clip(s). Saved.\n")

# ── Main loop ──────────────────────────────────────────────────────────
processed = 0

with tqdm(total=len(safe_pending), desc="VLM analysis") as pbar:
    for batch_start in range(0, len(safe_pending), BATCH_SIZE):
        batch_idx   = safe_pending[batch_start : batch_start + BATCH_SIZE]
        video_paths = [manifest.at[i, "path"] for i in batch_idx]

        try:
            results = analyze_batch(video_paths)
            for idx, analysis in zip(batch_idx, results):
                manifest.at[idx, "vlm_analysis"] = analysis
        except Exception as e:
            for idx in batch_idx:
                manifest.at[idx, "vlm_analysis"] = f"ERROR: {e}"
            print(f"\nError on batch {list(batch_idx)}: {e}")

        torch.cuda.empty_cache()

        prev_processed = processed
        processed += len(batch_idx)
        pbar.update(len(batch_idx))

        if processed // SAVE_EVERY > prev_processed // SAVE_EVERY:
            manifest.to_csv(MANIFEST_PATH, index=False)

# Final save
manifest.to_csv(MANIFEST_PATH, index=False)

done   = manifest["vlm_analysis"].notna().sum()
errors = manifest["vlm_analysis"].str.startswith("ERROR:", na=False).sum()
skips  = manifest["vlm_analysis"].str.startswith("SKIP:", na=False).sum()
print(f"\nDone. {done}/{total} clips processed  |  {errors} errors  |  {skips} permanent skips.")
print(f"Results saved → {MANIFEST_PATH}")
manifest.head(3)


Total clips : 13707
Already done: 13707  |  Remaining: 0
Re-labelled 2 legacy 'ERROR: video too long' → 'SKIP:'

Queued for processing: 0 clips


VLM analysis: 0it [00:00, ?it/s]


Done. 13707/13707 clips processed  |  2 errors  |  2 permanent skips.
Results saved → /mnt/Work/ML/Code/EmoRecVid/MELD.Raw/meld_vlm_manifest.csv


,sr_no,Utterance,Speaker,Emotion,Sentiment,Dialogue_ID,Utterance_ID,Season,Episode,StartTime,EndTime,split,path,vlm_analysis
0,1,also I was the point person on my companys tr...,Chandler,neutral,neutral,0,0,8,21,"00:16:16,059","00:16:21,731",train,/mnt/Work/ML/Code/EmoRecVid/MELD.Raw/train/tra...,The video begins with both individuals maintai...
1,2,You mustve had your hands full.,The Interviewer,neutral,neutral,0,1,8,21,"00:16:21,940","00:16:23,442",train,/mnt/Work/ML/Code/EmoRecVid/MELD.Raw/train/tra...,"The man begins with a neutral expression, his ..."
2,3,That I did. That I did.,Chandler,neutral,neutral,0,2,8,21,"00:16:23,442","00:16:26,389",train,/mnt/Work/ML/Code/EmoRecVid/MELD.Raw/train/tra...,The individual begins with a neutral expressio...


In [7]:

# ── Retry bad / empty / errored clips (loops until all are clean) ─────
# SKIP: entries (oversized / corrupt / unplayable) are never retried.
manifest = pd.read_csv(MANIFEST_PATH)


def _is_bad_output(val) -> bool:
    """True if a non-skip, non-error value is corrupted or too short."""
    if pd.isna(val):
        return False
    s = str(val)
    if s.startswith("ERROR:") or s.startswith("SKIP:"):
        return False
    words = s.strip().split()
    if len(words) <= 10:
        return True
    if words[0].lower() == "the" and words[-1].lower() == "the":
        return True
    return False


def get_retry_mask(df):
    is_skip  = df["vlm_analysis"].str.startswith("SKIP:", na=False)
    is_error = df["vlm_analysis"].str.startswith("ERROR:", na=False)
    is_empty = df["vlm_analysis"].isna()
    is_bad   = df["vlm_analysis"].map(_is_bad_output)
    return (is_error | is_empty | is_bad) & ~is_skip


pass_num = 0

while True:
    retry_mask = get_retry_mask(manifest)
    retry_idx  = list(manifest[retry_mask].index)

    is_skip  = manifest["vlm_analysis"].str.startswith("SKIP:", na=False)
    is_error = manifest["vlm_analysis"].str.startswith("ERROR:", na=False)
    is_empty = manifest["vlm_analysis"].isna()
    is_bad   = manifest["vlm_analysis"].map(_is_bad_output)

    print(f"\n── Pass {pass_num} ── "
          f"Empty: {is_empty.sum()}  |  Errored: {is_error.sum()}  |  "
          f"Bad: {is_bad.sum()}  |  Skipped: {is_skip.sum()}  |  "
          f"To retry: {len(retry_idx)}")

    if len(retry_idx) == 0:
        print("All clips are clean — done!")
        break

    # Before retrying, re-run playability check; permanently skip anything corrupt.
    newly_skipped = 0
    safe_retry = []
    for idx in retry_idx:
        vp = manifest.at[idx, "path"]
        dur, size_mb = get_video_info(vp)
        if dur > SKIP_DUR_S or size_mb > SKIP_SIZE_MB:
            reason = f"dur={dur:.1f}s" if dur > SKIP_DUR_S else f"size={size_mb:.1f}MB"
            manifest.at[idx, "vlm_analysis"] = f"SKIP: clip too large ({reason})"
            newly_skipped += 1
            continue
        ok, reason = is_playable(vp)
        if not ok:
            manifest.at[idx, "vlm_analysis"] = f"SKIP: unplayable ({reason})"
            newly_skipped += 1
            continue
        safe_retry.append(idx)

    if newly_skipped:
        manifest.to_csv(MANIFEST_PATH, index=False)
        print(f"  Permanently skipped {newly_skipped} unplayable/oversized clip(s).")

    if not safe_retry:
        print("  No retryable clips remain after playability check.")
        continue  # re-evaluate the while condition

    pass_num += 1
    processed = 0

    with tqdm(total=len(safe_retry), desc=f"Retry pass {pass_num}") as pbar:
        for batch_start in range(0, len(safe_retry), BATCH_SIZE):
            batch_idx   = safe_retry[batch_start : batch_start + BATCH_SIZE]
            video_paths = [manifest.at[i, "path"] for i in batch_idx]

            try:
                results = analyze_batch(video_paths)
                for idx, analysis in zip(batch_idx, results):
                    manifest.at[idx, "vlm_analysis"] = analysis
            except Exception as e:
                for idx in batch_idx:
                    manifest.at[idx, "vlm_analysis"] = f"ERROR: {e}"
                print(f"\nError on batch {list(batch_idx)}: {e}")

            torch.cuda.empty_cache()

            prev_processed = processed
            processed += len(batch_idx)
            pbar.update(len(batch_idx))

            if processed // SAVE_EVERY > prev_processed // SAVE_EVERY:
                manifest.to_csv(MANIFEST_PATH, index=False)

    manifest.to_csv(MANIFEST_PATH, index=False)

# Final summary
total  = len(manifest)
done   = manifest["vlm_analysis"].notna().sum()
errors = manifest["vlm_analysis"].str.startswith("ERROR:", na=False).sum()
skips  = manifest["vlm_analysis"].str.startswith("SKIP:", na=False).sum()
print(f"\nFinal: {done}/{total} processed  |  {errors} errors  |  {skips} permanent skips  |  {pass_num} retry pass(es).")
print(f"Results saved → {MANIFEST_PATH}")
manifest.head(3)



── Pass 0 ── Empty: 0  |  Errored: 2  |  Bad: 1  |  Skipped: 2  |  To retry: 3
  Permanently skipped 1 unplayable/oversized clip(s).


Retry pass 1:   0%|          | 0/2 [00:00<?, ?it/s]

qwen-vl-utils using torchvision to read video.
/mnt/Work/Environments/Ubuntu/Conda/envs/kaggle/lib/python3.12/site-packages/torchvision/io/_video_deprecation_warning.py:5: UserWarning: The video decoding and encoding capabilities of torchvision are deprecated from version 0.22 and will be removed in version 0.24. We recommend that you migrate to TorchCodec, where we'll consolidate the future decoding/encoding capabilities of PyTorch: https://github.com/pytorch/torchcodec
  warnings.warn(



── Pass 1 ── Empty: 0  |  Errored: 0  |  Bad: 0  |  Skipped: 3  |  To retry: 0
All clips are clean — done!

Final: 13707/13707 processed  |  0 errors  |  3 permanent skips  |  1 retry pass(es).
Results saved → /mnt/Work/ML/Code/EmoRecVid/MELD.Raw/meld_vlm_manifest.csv


,sr_no,Utterance,Speaker,Emotion,Sentiment,Dialogue_ID,Utterance_ID,Season,Episode,StartTime,EndTime,split,path,vlm_analysis
0,1,also I was the point person on my companys tr...,Chandler,neutral,neutral,0,0,8,21,"00:16:16,059","00:16:21,731",train,/mnt/Work/ML/Code/EmoRecVid/MELD.Raw/train/tra...,The video begins with both individuals maintai...
1,2,You mustve had your hands full.,The Interviewer,neutral,neutral,0,1,8,21,"00:16:21,940","00:16:23,442",train,/mnt/Work/ML/Code/EmoRecVid/MELD.Raw/train/tra...,"The man begins with a neutral expression, his ..."
2,3,That I did. That I did.,Chandler,neutral,neutral,0,2,8,21,"00:16:23,442","00:16:26,389",train,/mnt/Work/ML/Code/EmoRecVid/MELD.Raw/train/tra...,The individual begins with a neutral expressio...
